In [2]:
import pandas as pd
from pathlib import Path
import re

In [3]:
DATA_DIR = Path("./data/labeled")
OUTPUT_FILE = DATA_DIR / "Chile_all1_clean.csv"
PART_FILE_PATTERN = "Chile_all1_clean_labeled_*.csv"

In [4]:
def natural_part_key(path: Path) -> int:
    match = re.search(r"part(\d+)", path.stem)
    if not match:
        return 10**9
    return int(match.group(1))

In [5]:
def read_tripadvisor_data(path: str | Path) -> pd.DataFrame:
    data_path = Path(path)
    files = sorted(data_path.glob(PART_FILE_PATTERN), key=natural_part_key)
    if not files:
        raise FileNotFoundError(
            f"No se encontraron archivos con patron {PART_FILE_PATTERN} en {data_path}"
        )

    dataframes = [pd.read_csv(file_path, sep=";", encoding="utf-8") for file_path in files]
    return pd.concat(dataframes, ignore_index=True)

In [6]:
df_raw = read_tripadvisor_data(DATA_DIR)

In [6]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer

class SuggestionClassifier(nn.Module):
    def __init__(self, model_name="answerdotai/ModernBERT-base", num_classes=2):
        super().__init__()
        # Cargar el modelo base ModernBERT
        self.bert = AutoModel.from_pretrained(model_name)
        
        # 1. Congelar todos los parámetros de ModernBERT
        for param in self.bert.parameters():
            param.requires_grad = False
            
        # 2. Descongelar las últimas 5 capas de ModernBERT para Fine-Tuning
        num_layers_to_unfreeze = 2
        
        # En la estructura de ModernBERT, las capas están en bert.layers
        if hasattr(self.bert, 'layers'):
            for layer in self.bert.layers[-num_layers_to_unfreeze:]:
                for param in layer.parameters():
                    param.requires_grad = True
        elif hasattr(self.bert, 'encoder') and hasattr(self.bert.encoder, 'layer'):
            for layer in self.bert.encoder.layer[-num_layers_to_unfreeze:]:
                for param in layer.parameters():
                    param.requires_grad = True
                    
        # Descongelar la normalización final (si la tiene)
        if hasattr(self.bert, 'final_layer_norm'):
            for param in self.bert.final_layer_norm.parameters():
                param.requires_grad = True
            
        # 3. Añadir capas densas MÁS profundas
        hidden_size = self.bert.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )
        
    def forward(self, input_ids, attention_mask):
        # Obtener los outputs de BERT
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        
        # Usar Mean Pooling en lugar de solo agarrar [:, 0, :]
        # Da a la red una representación combinada de todo el contexto, lo que baja el error
        token_embeddings = outputs.last_hidden_state
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        mean_pooled = sum_embeddings / sum_mask
        
        # Pasar por las capas densas entrenables
        logits = self.classifier(mean_pooled)
        return logits

# Configurar el dispositivo para Mac M4 (Metal Performance Shaders - MPS)
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Optimizando para utilizar la GPU de Mac (MPS)")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
    print("Utilizando CPU")

# Instanciar el tokenizador y el modelo modificado
MODEL_ID = "answerdotai/ModernBERT-base" 
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = SuggestionClassifier(model_name=MODEL_ID).to(device)

print(f"\nDispositivo en uso: {device}")
print("Configuración de capas lista: Últimas 5 de BERT descongeladas + Clasificador denso profundo.")

Optimizando para utilizar la GPU de Mac (MPS)


Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

[transformers] ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
head.dense.weight | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Dispositivo en uso: mps
Configuración de capas lista: Últimas 5 de BERT descongeladas + Clasificador denso profundo.


In [7]:
df_raw.dropna(inplace=True)

In [9]:
df_raw.columns

Index(['review_text', 'title', 'language', 'language_detected_context',
       'language_detected_full', 'language_majority_lang',
       'language_majority_percentage', 'review_text_preprocessed',
       'razonamiento_corto', 'etiqueta', 'estado_etiquetado'],
      dtype='str')

In [11]:
df_raw.shape

(20896, 11)

In [12]:
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import numpy as np

# 1. Separar los datos por tipo de etiqueta
df_clean = df_raw.dropna(subset=['review_text', 'etiqueta']).copy()

# 2. Filtrar y muestrear
sugg_df = df_clean[df_clean['etiqueta'].str.upper() == 'SUGGESTION']
nonsugg_df = df_clean[df_clean['etiqueta'].str.upper() == 'NON-SUGGESTION']

# Tomar 500 sugerencias y 1000 no sugerencias para el entrenamiento 
num_sugg = min(1000, len(sugg_df))
num_nonsugg = min(1000, len(nonsugg_df))

train_sugg = sugg_df.sample(n=num_sugg, random_state=42)
train_nonsugg = nonsugg_df.sample(n=num_nonsugg, random_state=42)

df_train = pd.concat([train_sugg, train_nonsugg]).sample(frac=1, random_state=42).reset_index(drop=True)

# 3. Mapear a arreglos one-hot ( [1.0, 0.0] para NON-SUGGESTION, [0.0, 1.0] para SUGGESTION )
def map_label(label):
    if str(label).upper() == 'SUGGESTION':
        return [0.0, 1.0]
    return [1.0, 0.0]

labels = np.array(df_train['etiqueta'].apply(map_label).tolist())
texts = df_train['review_text'].tolist()

# 4. Crear Dataset y DataLoader para PyTorch
class TripAdvisorDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.float)
        }

# Crear DataLoader
train_dataset = TripAdvisorDataset(texts, labels, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

# 5. Configurar la función de pérdida y optimizador (VITAL AQUÍ)
# Calculamos un pos_weight para balancear las 1000 no-sug contra las 500 sug
weight_class_0 = num_sugg / num_nonsugg if num_nonsugg > 0 else 1.0 # Para la clase NON-SUGGESTION (más baja)
weight_class_1 = num_nonsugg / num_sugg if num_sugg > 0 else 1.0    # Para la clase SUGGESTION (más alta)
pos_weight = torch.tensor([weight_class_0, weight_class_1]).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# Alimentamos al optimizador SOLO los parámetros que requieren gradiente 
# (el clasificador nuevo + las últimas 5 capas de BERT)
trainable_params = filter(lambda p: p.requires_grad, model.parameters())

# Reducimos drásticamente el Learning Rate para el fine-tuning y evitar oscilaciones de loss
optimizer = optim.AdamW(trainable_params, lr=5e-5)

# 6. Bucle de Entrenamiento
EPOCHS = 12 # Al descongelar ModernBERT, el entrenamiento es más veloz, con 15 bastará posiblemente.

print(f"Iniciando entrenamiento con {len(df_train)} muestras ({num_sugg} SUGG, {num_nonsugg} NON-SUGG) y Learning Rate de 5e-5.")

model.train()
for epoch in range(EPOCHS):
    total_loss = 0
    for batch_idx, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        targets = batch['labels'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        if (batch_idx + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{EPOCHS} | Batch {batch_idx+1}/{len(train_loader)} | Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(train_loader)
    print(f"--- Epoch {epoch+1} completada | Loss Promedio: {avg_loss:.4f} ---\n")

print("Entrenamiento de múltiples capas finalizado.")

Iniciando entrenamiento con 2000 muestras (1000 SUGG, 1000 NON-SUGG) y Learning Rate de 5e-5.
Epoch 1/12 | Batch 10/125 | Loss: 0.6823
Epoch 1/12 | Batch 20/125 | Loss: 0.6685
Epoch 1/12 | Batch 30/125 | Loss: 0.6976
Epoch 1/12 | Batch 40/125 | Loss: 0.6929
Epoch 1/12 | Batch 50/125 | Loss: 0.6492
Epoch 1/12 | Batch 60/125 | Loss: 0.6324
Epoch 1/12 | Batch 70/125 | Loss: 0.6451
Epoch 1/12 | Batch 80/125 | Loss: 0.6359
Epoch 1/12 | Batch 90/125 | Loss: 0.5354
Epoch 1/12 | Batch 100/125 | Loss: 0.6722
Epoch 1/12 | Batch 110/125 | Loss: 0.5704
Epoch 1/12 | Batch 120/125 | Loss: 0.6880
--- Epoch 1 completada | Loss Promedio: 0.6398 ---

Epoch 2/12 | Batch 10/125 | Loss: 0.6190
Epoch 2/12 | Batch 20/125 | Loss: 0.5463
Epoch 2/12 | Batch 30/125 | Loss: 0.5458
Epoch 2/12 | Batch 40/125 | Loss: 0.5951
Epoch 2/12 | Batch 50/125 | Loss: 0.4035
Epoch 2/12 | Batch 60/125 | Loss: 0.5635
Epoch 2/12 | Batch 70/125 | Loss: 0.7881
Epoch 2/12 | Batch 80/125 | Loss: 0.6185
Epoch 2/12 | Batch 90/125 | Los

In [12]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import torch

# 1. Obtener los datos de prueba excluyendo los índices utilizados en el entrenamiento
test_sugg = sugg_df.drop(train_sugg.index)
test_nonsugg = nonsugg_df.drop(train_nonsugg.index)

df_test = pd.concat([test_sugg, test_nonsugg]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Datos de prueba preparados: {len(df_test)} muestras ({len(test_sugg)} SUGGESTION, {len(test_nonsugg)} NON-SUGGESTION).")

# 2. Mapear etiquetas y extraer textos usando la función map_label ya definida
test_labels = np.array(df_test['etiqueta'].apply(map_label).tolist())
test_texts = df_test['review_text'].tolist()

# 3. Crear Dataset y DataLoader para validación
test_dataset = TripAdvisorDataset(test_texts, test_labels, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# 4. Modo de evaluación de PyTorch
model.eval()

all_preds = []
all_targets = []

print("Iniciando evaluación del modelo sobre los datos no vistos...")

with torch.no_grad(): # No necesitamos calcular gradientes en validación/test
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        targets = batch['labels'].to(device)
        
        outputs = model(input_ids, attention_mask)
        
        # Nuestros tensores target son [1,0] o [0,1]. La posición con valor más alto indica la clase
        # 0 = NON-SUGGESTION (predicción para [1,0])
        # 1 = SUGGESTION (predicción para [0,1])
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        true_labels = torch.argmax(targets, dim=1).cpu().numpy()
        
        all_preds.extend(preds)
        all_targets.extend(true_labels)

# 5. Calcular y mostrar las métricas
# La clase 1 (SUGGESTION) es la positiva para Precision, Recall y F1 Score
acc = accuracy_score(all_targets, all_preds)
prec = precision_score(all_targets, all_preds, pos_label=1, zero_division=0)
rec = recall_score(all_targets, all_preds, pos_label=1, zero_division=0)
f1 = f1_score(all_targets, all_preds, pos_label=1, zero_division=0)

print("\n=== Resultados de Evaluación (Datos de Test) ===")
print(f"Accuracy  : {acc:.4f} (Porcentaje de clasificaciones correctas en general)")
print(f"F1 Score  : {f1:.4f} (Media armónica entre Precision y Recall)")
print(f"Recall    : {rec:.4f} (De todas las sugerencias reales, ¿qué porcentaje detectó el modelo?)")
print(f"Precision : {prec:.4f} (De todas las que clasificó como sugerencia, ¿qué porcentaje realmente lo era?)")

print("\n--- Reporte Detallado de Sklearn ---")
print(classification_report(
    all_targets, 
    all_preds, 
    target_names=['NON-SUGGESTION (0)', 'SUGGESTION (1)'],
    zero_division=0
))

Datos de prueba preparados: 13901 muestras (0 SUGGESTION, 13901 NON-SUGGESTION).
Iniciando evaluación del modelo sobre los datos no vistos...


TypeError: argmax(): argument 'input' (position 1) must be Tensor, not SequenceClassifierOutput

In [15]:
def classify_text(text):
    """
    Toma un texto de prueba, lo tokeniza como en el entrenamiento y lo
    pasa por el modelo para retornar la etiqueta predicha por la IA.
    """
    model.eval() # Asegurarse de que esté en modo evaluación
    
    # 1. Tokenización (misma configuración que en TripAdvisorDataset)
    encoding = tokenizer(
        text,
        max_length=512,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    
    # 2. Mover tensores al dispositivo (GPU MPS / CUDA / CPU)
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    
    # 3. Inferencia
    with torch.no_grad():
        outputs = model(input_ids, attention_mask)
        # argmax para obtener si es 0 (NON-SUGGESTION) o 1 (SUGGESTION)
        prediction_idx = torch.argmax(outputs, dim=1).item()
        
    # 4. Retornar el nombre de la etiqueta
    labels_map = {0: "NON-SUGGESTION", 1: "SUGGESTION"}
    return labels_map[prediction_idx]

# ==========================================
# Etiquetando 250 comentarios de la data original
# ==========================================

# Extraemos 250 comentarios aleatorios de la data original asegurándonos de que tengan texto
sample_df = df_raw.dropna(subset=['review_text']).sample(n=250, random_state=42)
sample_texts = sample_df['review_text'].tolist()

print(f"=== Etiquetando {len(sample_texts)} comentarios aleatorios ===")

resultados_ia = []
for i, review in enumerate(sample_texts):
    pred_label = classify_text(review)
    resultados_ia.append({'review_text': review, 'prediccion': pred_label})
    
    # Imprimimos los primeros 5 ejemplos en consola para verificar
    if i < 5:
        # Mostramos los primeros 120 caracteres para que no ocupe tanto espacio en pantalla
        print(f"[{i+1}] Texto: \"{review[:120]}...\"\n--> Predicción: {pred_label}\n")

print(f"... y {len(sample_texts) - 5} comentarios más procesados exitosamente.")

# Convertimos los resultados a un DataFrame para revisar y manipular
df_resultados_ia = pd.DataFrame(resultados_ia)

print("\n--- Resumen de Etiquetas Asignadas ---")
print(df_resultados_ia['prediccion'].value_counts())

# Mostramos una tabla con los primeros 10 resultados
df_resultados_ia.head(10)

=== Etiquetando 250 comentarios aleatorios ===
[1] Texto: "Nice lake, but the summer view is gorgeous and makes you wonder why see it on a winter day. Good option for a free after..."
--> Predicción: NON-SUGGESTION

[2] Texto: "Located beside the river Calle Calle, this is largely a fish and sea food market, with a few stalls selling cheese. The ..."
--> Predicción: NON-SUGGESTION

[3] Texto: "If you are interested in the art of indigenous people you should go there. They have a party of the museum which is call..."
--> Predicción: NON-SUGGESTION

[4] Texto: "We had a great day in Valparaiso with Christian and Oscar. After a short visit to Vina del Mar, we drove on to Valpo for..."
--> Predicción: NON-SUGGESTION

[5] Texto: "We wish we could give this a good review but cannot since we did not get the tour we booked. We booked the premium tour ..."
--> Predicción: SUGGESTION

... y 245 comentarios más procesados exitosamente.

--- Resumen de Etiquetas Asignadas ---
prediccion
NON-SUGGES

,review_text,prediccion
0,"Nice lake, but the summer view is gorgeous and...",NON-SUGGESTION
1,"Located beside the river Calle Calle, this is ...",NON-SUGGESTION
2,If you are interested in the art of indigenous...,NON-SUGGESTION
3,We had a great day in Valparaiso with Christia...,NON-SUGGESTION
4,We wish we could give this a good review but c...,SUGGESTION
5,Hicimos la excursión de la tarde de ayer y fue...,NON-SUGGESTION
6,"The churches of Chiloe are beautiful, and thei...",SUGGESTION
7,We did the day-long kayaking excursion in mid-...,NON-SUGGESTION
8,"Olá Transfer took place very quietly, with goo...",NON-SUGGESTION
9,We booked two full days (> $1K USD) for wine t...,SUGGESTION


In [16]:
df_resultados_ia.to_csv("resultados_ia.csv", index=False, encoding="utf-8-sig")

In [17]:
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from torch import optim

# ── 1. MODELO: usar AutoModelForSequenceClassification en vez de cabeza custom ──
from transformers import AutoModelForSequenceClassification

MODEL_ID = "answerdotai/ModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=2,
    ignore_mismatched_sizes=True
)

# Congelar todo excepto las últimas 3 capas + clasificador
for param in model.parameters():
    param.requires_grad = False

"""for layer in model.model.layers[-3:]:  # ModernBERT usa model.layers
    for param in layer.parameters():
        param.requires_grad = True"""

for param in model.classifier.parameters():
    param.requires_grad = True

device = torch.device("mps" if torch.backends.mps.is_available() 
                       else "cuda" if torch.cuda.is_available() 
                       else "cpu")
model = model.to(device)
print(f"Dispositivo: {device}")


# ── 2. DATASET: limpiar FPs antes de entrenar ──
# Solo usar como SUGGESTION las filas que OLMo etiquetó Y que 
# pasaron revisión manual (eliminar los FPs conocidos)
FP_INDICES_TO_EXCLUDE = {
    # Añadir aquí los índices de FPs identificados en el análisis
    # ej: índices donde OLMo etiquetó SUGGESTION incorrectamente
}

df_clean = df_raw[~df_raw.index.isin(FP_INDICES_TO_EXCLUDE)].copy()
df_clean = df_clean[df_clean['etiqueta'].notna()]

sugg_df    = df_clean[df_clean['etiqueta'] == 'SUGGESTION']
nonsugg_df = df_clean[df_clean['etiqueta'] == 'NON-SUGGESTION']

# Balanceo: usar TODOS los SUGGESTION reales + misma cantidad de NON-SUGGESTION
# No inflar artificialmente — respeta la proporción real con oversampling moderado
n_sugg = len(sugg_df)
train_sugg    = sugg_df.sample(n=min(n_sugg, len(sugg_df) - 300), random_state=42)
train_nonsugg = nonsugg_df.sample(n=min(n_sugg * 3, len(nonsugg_df)), random_state=42)
# Ratio 1:4 (sugg:non-sugg) — más cercano a la realidad que 1:1

df_train = pd.concat([train_sugg, train_nonsugg]).sample(frac=1, random_state=42)

print(f"Train: {len(train_sugg)} SUGGESTION | {len(train_nonsugg)} NON-SUGGESTION")


# ── 3. DATASET CLASS: targets escalares para CrossEntropyLoss ──
def map_label(label):
    return 1 if label == 'SUGGESTION' else 0

class TripAdvisorDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)  # escalar, no one-hot
        }

texts  = df_train['review_text'].tolist()
labels = df_train['etiqueta'].apply(map_label).tolist()

train_dataset = TripAdvisorDataset(texts, labels, tokenizer)
train_loader  = DataLoader(train_dataset, batch_size=16, shuffle=True)


# ── 4. ENTRENAMIENTO: CrossEntropyLoss + LR bajo + scheduler + early stopping ──
# pos_weight para la clase minoritaria (SUGGESTION)
pos_weight = torch.tensor([1.0, len(train_nonsugg) / len(train_sugg)]).to(device)
criterion  = nn.CrossEntropyLoss(weight=pos_weight)

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=2e-5,          # ← reducido de 5e-5 a 2e-5
    weight_decay=0.01
)

EPOCHS = 14 # ← reducido de 12 a 5 para evitar overfitting
total_steps = len(train_loader) * EPOCHS

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,  # 10% de warmup
    num_training_steps=total_steps
)

best_f1   = 0.0
patience  = 2
no_improve = 0

model.train()
for epoch in range(EPOCHS):
    total_loss = 0
    for batch_idx, batch in enumerate(train_loader):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        targets        = batch['labels'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss    = criterion(outputs.logits, targets)  # targets escalares
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # gradient clipping
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f}")

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Dispositivo: mps
Train: 1099 SUGGESTION | 4197 NON-SUGGESTION
Epoch 1/14 | Loss: 0.6229
Epoch 2/14 | Loss: 0.5032
Epoch 3/14 | Loss: 0.4278
Epoch 4/14 | Loss: 0.3873
Epoch 5/14 | Loss: 0.3583
Epoch 6/14 | Loss: 0.3237
Epoch 7/14 | Loss: 0.2969
Epoch 8/14 | Loss: 0.2675
Epoch 9/14 | Loss: 0.2353
Epoch 10/14 | Loss: 0.2065
Epoch 11/14 | Loss: 0.1808
Epoch 12/14 | Loss: 0.1626
Epoch 13/14 | Loss: 0.1495
Epoch 14/14 | Loss: 0.1403


In [15]:
for epoch in range(3):
    total_loss = 0
    for batch_idx, batch in enumerate(train_loader):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        targets        = batch['labels'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss    = criterion(outputs.logits, targets)  # targets escalares
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # gradient clipping
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f}")

KeyboardInterrupt: 

In [18]:
# ── CELDA DE EVALUACIÓN Y TEST ──────────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)

# ── 1. PREPARAR EL TEST SET ─────────────────────────────────────────────────
# Excluir filas usadas en entrenamiento para evitar data leakage
train_indices = set(df_train.index)

df_test = df_clean[~df_clean.index.isin(train_indices)].copy()
test_sugg    = df_test[df_test['etiqueta'] == 'SUGGESTION']
test_nonsugg = df_test[df_test['etiqueta'] == 'NON-SUGGESTION']

print(f"Test set: {len(test_sugg)} SUGGESTION | {len(test_nonsugg)} NON-SUGGESTION")
print(f"Tasa real de SUGGESTION en test: {len(test_sugg)/len(df_test):.1%}\n")

test_texts  = df_test['review_text'].tolist()
test_labels = df_test['etiqueta'].apply(map_label).tolist()

test_dataset = TripAdvisorDataset(test_texts, test_labels, tokenizer)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

# ── 2. INFERENCIA SOBRE EL TEST SET ─────────────────────────────────────────
model.eval()
all_preds   = []
all_targets = []
all_probs   = []  # probabilidades para análisis de threshold

print("Evaluando sobre el test set...")
with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        targets        = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits  = outputs.logits                             # shape: (batch, 2)
        probs   = torch.softmax(logits, dim=1)[:, 1]        # prob de SUGGESTION
        preds   = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(targets.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

# ── 3. MÉTRICAS PRINCIPALES ──────────────────────────────────────────────────
acc  = accuracy_score(all_targets, all_preds)
prec = precision_score(all_targets, all_preds, pos_label=1, zero_division=0)
rec  = recall_score(all_targets, all_preds, pos_label=1, zero_division=0)
f1   = f1_score(all_targets, all_preds, pos_label=1, zero_division=0)

print("=" * 55)
print("  RESULTADOS DE EVALUACIÓN — ModernBERT fine-tuned")
print("=" * 55)
print(f"  Accuracy  : {acc:.4f}")
print(f"  Precision : {prec:.4f}  (de los que predice SUGGESTION, ¿cuántos son reales?)")
print(f"  Recall    : {rec:.4f}  (de los SUGGESTION reales, ¿cuántos detecta?)")
print(f"  F1-score  : {f1:.4f}  (balance precisión-recall)")
print("=" * 55)

# ── 4. MATRIZ DE CONFUSIÓN ───────────────────────────────────────────────────
cm = confusion_matrix(all_targets, all_preds)
tn, fp, fn, tp = cm.ravel()

print(f"\n  Matriz de Confusión:")
print(f"  {'':20s}  Pred NON-SUGG  Pred SUGG")
print(f"  {'Real NON-SUGG':20s}  {tn:13d}  {fp:9d}  ← Falsos Positivos: {fp}")
print(f"  {'Real SUGG':20s}  {fn:13d}  {tp:9d}  ← Falsos Negativos: {fn}")

# ── 5. REPORTE DETALLADO ─────────────────────────────────────────────────────
print(f"\n  Reporte Sklearn:")
print(classification_report(
    all_targets, all_preds,
    target_names=['NON-SUGGESTION', 'SUGGESTION'],
    zero_division=0
))

# ── 6. ANÁLISIS DE THRESHOLD ─────────────────────────────────────────────────
# Si la precisión es baja, subir el threshold mejora precisión a costa de recall
print("  Análisis de threshold (probabilidad mínima para clasificar como SUGGESTION):")
print(f"  {'Threshold':>10}  {'Precision':>10}  {'Recall':>8}  {'F1':>8}  {'SUGG pred':>10}")
for thresh in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    preds_t = [1 if p >= thresh else 0 for p in all_probs]
    p = precision_score(all_targets, preds_t, pos_label=1, zero_division=0)
    r = recall_score(all_targets, preds_t, pos_label=1, zero_division=0)
    f = f1_score(all_targets, preds_t, pos_label=1, zero_division=0)
    n_sugg_pred = sum(preds_t)
    marker = " ← default" if thresh == 0.5 else ""
    print(f"  {thresh:>10.1f}  {p:>10.4f}  {r:>8.4f}  {f:>8.4f}  {n_sugg_pred:>10d}{marker}")

# ── 7. FUNCIÓN DE INFERENCIA INDIVIDUAL ─────────────────────────────────────
def classify_text(text: str, threshold: float = 0.5) -> dict:
    """
    Clasifica un texto individual.
    Retorna label, probabilidad de SUGGESTION y confianza.
    """
    model.eval()
    encoding = tokenizer(
        text,
        max_length=512,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    input_ids      = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs   = torch.softmax(outputs.logits, dim=1)[0]
        prob_suggestion = probs[1].item()

    label = "SUGGESTION" if prob_suggestion >= threshold else "NON-SUGGESTION"
    return {
        "label":            label,
        "prob_suggestion":  round(prob_suggestion, 4),
        "prob_non_sugg":    round(probs[0].item(), 4),
        "threshold_usado":  threshold,
    }

# ── 8. EJEMPLOS DE INFERENCIA INDIVIDUAL ────────────────────────────────────
print("\n  Ejemplos de inferencia individual:")
ejemplos = [
    "The bed was hard and the service was slow.",                              # NON-SUGGESTION
    "They should add an English translation option for international visitors.", # SUGGESTION
    "I highly recommend doing the tour with Pedro, he was amazing!",           # NON-SUGGESTION
    "It would be great if they included a map at the entrance.",               # SUGGESTION
    "Beautiful place, definitely visit in summer.",                            # NON-SUGGESTION
]

for texto in ejemplos:
    result = classify_text(texto, threshold=0.5)
    bar = "█" * int(result['prob_suggestion'] * 20)
    print(f"\n  Texto : {texto[:80]}...")
    print(f"  Label : {result['label']}  (p={result['prob_suggestion']:.3f})  [{bar:<20}]")

Test set: 300 SUGGESTION | 15300 NON-SUGGESTION
Tasa real de SUGGESTION en test: 1.9%

Evaluando sobre el test set...
  RESULTADOS DE EVALUACIÓN — ModernBERT fine-tuned
  Accuracy  : 0.8828
  Precision : 0.1086  (de los que predice SUGGESTION, ¿cuántos son reales?)
  Recall    : 0.7067  (de los SUGGESTION reales, ¿cuántos detecta?)
  F1-score  : 0.1882  (balance precisión-recall)

  Matriz de Confusión:
                        Pred NON-SUGG  Pred SUGG
  Real NON-SUGG                 13559       1741  ← Falsos Positivos: 1741
  Real SUGG                        88        212  ← Falsos Negativos: 88

  Reporte Sklearn:
                precision    recall  f1-score   support

NON-SUGGESTION       0.99      0.89      0.94     15300
    SUGGESTION       0.11      0.71      0.19       300

      accuracy                           0.88     15600
     macro avg       0.55      0.80      0.56     15600
  weighted avg       0.98      0.88      0.92     15600

  Análisis de threshold (probabilidad

In [9]:
df_labeled = df_raw.copy()
df_labeled.columns

Index(['review_text', 'title', 'language', 'language_detected_context',
       'language_detected_full', 'language_majority_lang',
       'language_majority_percentage', 'review_text_preprocessed',
       'razonamiento_corto', 'etiqueta', 'estado_etiquetado'],
      dtype='str')

In [17]:
# ═══════════════════════════════════════════════════════════════════
#  ModernBERT — Fine-tuning para clasificación de sugerencias
#  Salida: [NON-SUGGESTION prob, SUGGESTION prob]
#  Dataset: ~1300 SUGGESTION / ~18000 NON-SUGGESTION
# ═══════════════════════════════════════════════════════════════════

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from torch import optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split

# ───────────────────────────────────────────────────────────────────
#  0. CONFIGURACIÓN GLOBAL
# ───────────────────────────────────────────────────────────────────
MODEL_ID   = "answerdotai/ModernBERT-base"
MAX_LEN    = 512
BATCH_SIZE = 16
EPOCHS     = 4
LR         = 2e-5
SEED       = 42

# Etiquetas: índice 0 = NON-SUGGESTION, índice 1 = SUGGESTION
# → softmax([logit_0, logit_1])[0] = prob NON-SUGGESTION
# → softmax([logit_0, logit_1])[1] = prob SUGGESTION
LABEL2ID = {"NON-SUGGESTION": 0, "SUGGESTION": 1}
ID2LABEL = {0: "NON-SUGGESTION", 1: "SUGGESTION"}

torch.manual_seed(SEED)
device = torch.device(
    "mps"  if torch.backends.mps.is_available()  else
    "cuda" if torch.cuda.is_available()           else
    "cpu"
)
print(f"Dispositivo: {device}")


# ───────────────────────────────────────────────────────────────────
#  1. CARGA Y SPLIT DE DATOS
#     df_labeled debe tener columnas: review_text | etiqueta
#     etiqueta ∈ {"SUGGESTION", "NON-SUGGESTION"}
# ───────────────────────────────────────────────────────────────────
# Reemplaza esta línea por tu DataFrame ya cargado:
# df_labeled = pd.read_csv("tu_dataset.csv", sep=";")

df_labeled = df_labeled[df_labeled["etiqueta"].isin(LABEL2ID)].copy()
df_labeled["label_id"] = df_labeled["etiqueta"].map(LABEL2ID)

print(f"Total muestras: {len(df_labeled)}")
print(df_labeled["etiqueta"].value_counts())

# Split estratificado 80/10/10 (train / val / test)
df_train, df_temp = train_test_split(
    df_labeled, test_size=0.20,
    stratify=df_labeled["label_id"], random_state=SEED
)
df_non_sug = df_train[df_train["etiqueta"] == "NON-SUGGESTION"].head(15000)

# 2. Filtramos todas las de "SUGGESTION"
df_sug = df_train[df_train["etiqueta"] == "SUGGESTION"]

# 3. Concatenamos ambas partes verticalmente
df_train = pd.concat([df_non_sug, df_sug], ignore_index=True)
df_val, df_test = train_test_split(
    df_temp, test_size=0.50,
    stratify=df_temp["label_id"], random_state=SEED
)

print(f"\nTrain : {len(df_train)}  "
      f"({df_train['etiqueta'].value_counts().to_dict()})")
print(f"Val   : {len(df_val)}  "
      f"({df_val['etiqueta'].value_counts().to_dict()})")
print(f"Test  : {len(df_test)}  "
      f"({df_test['etiqueta'].value_counts().to_dict()})")


# ───────────────────────────────────────────────────────────────────
#  2. DATASET
# ───────────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

class SuggestionDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts  = [str(t) for t in texts]
        self.labels = list(labels)           # enteros: 0 ó 1

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            max_length=MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),       # (MAX_LEN,)
            "attention_mask": enc["attention_mask"].squeeze(0),  # (MAX_LEN,)
            "label":          torch.tensor(self.labels[idx], dtype=torch.long),
        }

train_dataset = SuggestionDataset(
    df_train["review_text"].tolist(),
    df_train["label_id"].tolist()
)
val_dataset = SuggestionDataset(
    df_val["review_text"].tolist(),
    df_val["label_id"].tolist()
)
test_dataset = SuggestionDataset(
    df_test["review_text"].tolist(),
    df_test["label_id"].tolist()
)

# WeightedRandomSampler: sobre-muestrea SUGGESTION en cada batch
#   → el modelo ve sugerencias con suficiente frecuencia sin duplicar
#      artificialmente el dataset completo
class_counts  = df_train["label_id"].value_counts().sort_index().tolist()
class_weights = [1.0 / c for c in class_counts]   # [w_nonsugg, w_sugg]
sample_weights = [class_weights[lbl] for lbl in df_train["label_id"]]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler)
val_loader   = DataLoader(val_dataset,   batch_size=32,          shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=32,          shuffle=False)


# ───────────────────────────────────────────────────────────────────
#  3. MODELO
# ───────────────────────────────────────────────────────────────────
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=2,                    # [NON-SUGGESTION, SUGGESTION]
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
)

# Congelar todo ModernBERT
for param in model.parameters():
    param.requires_grad = False

# Descongelar últimas 3 capas transformer
for layer in model.model.layers[-3:]:
    for param in layer.parameters():
        param.requires_grad = True

# Descongelar cabeza de clasificación (siempre entrenable)
for param in model.classifier.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"\nParámetros entrenables: {trainable:,} / {total:,} "
      f"({100*trainable/total:.1f}%)")

model = model.to(device)


# ───────────────────────────────────────────────────────────────────
#  4. FUNCIÓN DE PÉRDIDA CON PESO A LA CLASE MINORITARIA
#     ratio = 18000/1300 ≈ 13.8 → el modelo penaliza 13.8× más
#     un FN (perder una sugerencia real) que un FP
# ───────────────────────────────────────────────────────────────────
n_nonsugg = (df_train["label_id"] == 0).sum()
n_sugg    = (df_train["label_id"] == 1).sum()
ratio     = n_nonsugg / n_sugg

class_weight_tensor = torch.tensor(
    [1.0, ratio],   # índice 0 = NON-SUGG, índice 1 = SUGG
    dtype=torch.float
).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weight_tensor)
print(f"Peso clase SUGGESTION en CrossEntropy: {ratio:.2f}×")


# ───────────────────────────────────────────────────────────────────
#  5. OPTIMIZADOR Y SCHEDULER
# ───────────────────────────────────────────────────────────────────
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR,
    weight_decay=0.01,
)

total_steps   = len(train_loader) * EPOCHS
warmup_steps  = total_steps // 10

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)


# ───────────────────────────────────────────────────────────────────
#  6. BUCLE DE ENTRENAMIENTO CON EARLY STOPPING SOBRE F1 DE VAL
# ───────────────────────────────────────────────────────────────────
def evaluate(loader):
    """Retorna preds, targets, probs y métricas sobre un DataLoader."""
    model.eval()
    all_preds, all_targets, all_probs = [], [], []

    with torch.no_grad():
        for batch in loader:
            ids  = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            lbls = batch["label"].to(device)

            logits = model(input_ids=ids, attention_mask=mask).logits
            probs  = torch.softmax(logits, dim=1)   # shape (B, 2)
            preds  = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(lbls.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())   # lista de [p0, p1]

    f1  = f1_score(all_targets, all_preds, pos_label=1, zero_division=0)
    prec = precision_score(all_targets, all_preds, pos_label=1, zero_division=0)
    rec  = recall_score(all_targets, all_preds, pos_label=1, zero_division=0)

    return all_preds, all_targets, np.array(all_probs), {
        "f1": f1, "precision": prec, "recall": rec
    }


best_val_f1  = 0.0
patience     = 2
no_improve   = 0
best_weights = None

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    for batch in train_loader:
        ids     = batch["input_ids"].to(device)
        mask    = batch["attention_mask"].to(device)
        targets = batch["label"].to(device)

        optimizer.zero_grad()
        logits = model(input_ids=ids, attention_mask=mask).logits
        loss   = criterion(logits, targets)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    # Validación al final de cada época
    _, _, _, val_metrics = evaluate(val_loader)
    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Loss: {avg_loss:.4f} | "
        f"Val F1: {val_metrics['f1']:.4f} | "
        f"P: {val_metrics['precision']:.4f} | "
        f"R: {val_metrics['recall']:.4f}"
    )

    # Early stopping: guarda los pesos del mejor F1 en validación
    if val_metrics["f1"] > best_val_f1:
        best_val_f1  = val_metrics["f1"]
        best_weights = {k: v.clone() for k, v in model.state_dict().items()}
        no_improve   = 0
        print(f"  ✓ Mejor F1 en validación: {best_val_f1:.4f} — pesos guardados")
    else:
        no_improve += 1
        print(f"  Sin mejora ({no_improve}/{patience})")
        if no_improve >= patience:
            print("  Early stopping activado.")
            break

# Restaurar los mejores pesos antes de evaluar en test
model.load_state_dict(best_weights)
print(f"\nPesos restaurados al mejor F1 de validación: {best_val_f1:.4f}")


# ───────────────────────────────────────────────────────────────────
#  7. EVALUACIÓN FINAL EN TEST SET
# ───────────────────────────────────────────────────────────────────
preds, targets, probs, test_metrics = evaluate(test_loader)

# probs[:, 0] = probabilidad NON-SUGGESTION
# probs[:, 1] = probabilidad SUGGESTION
prob_nonsugg = probs[:, 0]
prob_sugg    = probs[:, 1]

print("\n" + "═" * 55)
print("  EVALUACIÓN FINAL — TEST SET")
print("═" * 55)
print(f"  F1-score  : {test_metrics['f1']:.4f}")
print(f"  Precision : {test_metrics['precision']:.4f}")
print(f"  Recall    : {test_metrics['recall']:.4f}")

cm = confusion_matrix(targets, preds)
tn, fp, fn, tp = cm.ravel()
print(f"\n  Matriz de Confusión:")
print(f"  {'':22s}  Pred NON-SUGG  Pred SUGG")
print(f"  {'Real NON-SUGG':22s}  {tn:13d}  {fp:9d}")
print(f"  {'Real SUGG':22s}  {fn:13d}  {tp:9d}")

print(f"\n  Reporte Sklearn:")
print(classification_report(
    targets, preds,
    target_names=["NON-SUGGESTION", "SUGGESTION"],
    zero_division=0
))


# ───────────────────────────────────────────────────────────────────
#  8. ANÁLISIS DE THRESHOLD
#     Con datos muy desbalanceados, el threshold óptimo rara vez es 0.5
# ───────────────────────────────────────────────────────────────────
print("  Análisis de threshold sobre el test set:")
print(f"  {'Threshold':>10}  {'Precision':>10}  {'Recall':>8}  "
      f"{'F1':>8}  {'SUGG pred':>10}")

for thresh in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    preds_t = (prob_sugg >= thresh).astype(int)
    p = precision_score(targets, preds_t, pos_label=1, zero_division=0)
    r = recall_score(targets, preds_t, pos_label=1, zero_division=0)
    f = f1_score(targets, preds_t, pos_label=1, zero_division=0)
    n = preds_t.sum()
    marker = " ← default" if thresh == 0.5 else ""
    print(f"  {thresh:>10.1f}  {p:>10.4f}  {r:>8.4f}  {f:>8.4f}  {n:>10d}{marker}")


# ───────────────────────────────────────────────────────────────────
#  9. INFERENCIA INDIVIDUAL
#     Retorna el vector de probabilidades completo [p_nonsugg, p_sugg]
# ───────────────────────────────────────────────────────────────────
def classify(text: str, threshold: float = 0.5) -> dict:
    """
    Clasifica un comentario y retorna probabilidades para ambas clases.

    Retorna:
        {
          "label":          "SUGGESTION" | "NON-SUGGESTION",
          "probabilities":  {"NON-SUGGESTION": float, "SUGGESTION": float},
          "threshold":      float
        }
    """
    model.eval()
    enc = tokenizer(
        str(text),
        max_length=MAX_LEN,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )
    input_ids      = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)

    with torch.no_grad():
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        probs  = torch.softmax(logits, dim=1)[0].cpu().numpy()

    # probs[0] = NON-SUGGESTION, probs[1] = SUGGESTION
    label = "SUGGESTION" if probs[1] >= threshold else "NON-SUGGESTION"
    return {
        "label": label,
        "probabilities": {
            "NON-SUGGESTION": round(float(probs[0]), 4),
            "SUGGESTION":     round(float(probs[1]), 4),
        },
        "threshold": threshold,
    }


# ── Ejemplos ────────────────────────────────────────────────────────
ejemplos = [
    ("The bed was hard and the room smelled bad.",
     "NON-SUGGESTION esperado"),
    ("They should add an English translation option for visitors.",
     "SUGGESTION esperado"),
    ("Amazing tour with Pedro, I highly recommend him!",
     "NON-SUGGESTION esperado (consejo a turistas)"),
    ("It would be great if they included a map at the entrance.",
     "SUGGESTION esperado"),
    ("Beautiful place, definitely visit in summer.",
     "NON-SUGGESTION esperado"),
]

print("\n  Ejemplos de inferencia individual (threshold=0.5):")
for texto, esperado in ejemplos:
    result = classify(texto)
    p_sugg    = result["probabilities"]["SUGGESTION"]
    p_nonsugg = result["probabilities"]["NON-SUGGESTION"]
    bar       = "█" * int(p_sugg * 20)
    print(f"\n  {esperado}")
    print(f"  Texto  : {texto[:75]}")
    print(f"  Label  : {result['label']}")
    print(f"  p(NON) : {p_nonsugg:.4f}   p(SUGG) : {p_sugg:.4f}  [{bar:<20}]")

Dispositivo: mps
Total muestras: 20896
etiqueta
NON-SUGGESTION    19497
SUGGESTION         1399
Name: count, dtype: int64

Train : 16119  ({'NON-SUGGESTION': 15000, 'SUGGESTION': 1119})
Val   : 2090  ({'NON-SUGGESTION': 1950, 'SUGGESTION': 140})
Test  : 2090  ({'NON-SUGGESTION': 1950, 'SUGGESTION': 140})


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Parámetros entrenables: 15,046,658 / 149,606,402 (10.1%)
Peso clase SUGGESTION en CrossEntropy: 13.40×
Epoch 1/4 | Loss: 0.2200 | Val F1: 0.2351 | P: 0.1352 | R: 0.9000
  ✓ Mejor F1 en validación: 0.2351 — pesos guardados
Epoch 2/4 | Loss: 0.1224 | Val F1: 0.3149 | P: 0.1940 | R: 0.8357
  ✓ Mejor F1 en validación: 0.3149 — pesos guardados
Epoch 3/4 | Loss: 0.0703 | Val F1: 0.3894 | P: 0.2588 | R: 0.7857
  ✓ Mejor F1 en validación: 0.3894 — pesos guardados
Epoch 4/4 | Loss: 0.0501 | Val F1: 0.3849 | P: 0.2572 | R: 0.7643
  Sin mejora (1/2)

Pesos restaurados al mejor F1 de validación: 0.3894

═══════════════════════════════════════════════════════
  EVALUACIÓN FINAL — TEST SET
═══════════════════════════════════════════════════════
  F1-score  : 0.3797
  Precision : 0.2577
  Recall    : 0.7214

  Matriz de Confusión:
                          Pred NON-SUGG  Pred SUGG
  Real NON-SUGG                    1659        291
  Real SUGG                          39        101

  Reporte Sklearn